# GPT-2 Joke Arena v7 — original rolling telephone

This generates **160 real GPT-2 stories**.

## Standard — 80 stories

10 prompts × 8 stories:

- 2 GPT-2 Small
- 2 GPT-2 Medium
- 2 GPT-2 Large
- 2 GPT-2 XL

Each Standard story is a **single normal autocomplete call**:

- natural EOS enabled
- maximum displayed length = 300 words
- seed stream begins at **1337**
- temperature `0.95`
- top-p `0.95`
- top-k `50`

## Telephone — 80 stories

Same 10 prompts × same 8 model/replicate structure.

Telephone now uses the **same rolling self-continuation algorithm as the original 800-word bakeoff notebook**:

1. Start with the prompt.
2. Generate a chunk.
3. Append it to the accumulated story.
4. Feed the accumulated story back into GPT-2.
5. Once the accumulated text exceeds the model's context budget, keep only the newest context tokens.
6. **Suppress EOS**, just like the original funny run.
7. Repeat until at least 300 visible words exist, then cut the displayed story to exactly the first 300 words.

### Critical compatibility detail

Internally, the chunk scheduler still behaves as though the target were **800 words**, exactly like the original notebook. We merely stop after enough text exists to display the first 300 words.

That preserves the original 256-token chunk schedule and therefore lets the first Telephone sample for GPT-2 Small/Large reproduce the beginning of the original funny 800-word generations.

## Seed

For each model:

- Standard gets its own RNG stream seeded once at **1337**.
- Telephone gets a separate RNG stream seeded once at **1337**.

Thus Telephone #1 for the first prompt begins from exactly the same RNG state as the original bakeoff, while Telephone #2 and later stories continue the seeded stream and are distinct.


**v7 robustness fix:** normal rolling chunks remain exactly unchanged. Only a zero-visible-text chunk triggers a fallback retry with EOS stopping explicitly disabled.

In [ ]:
!pip -q install -U "transformers>=4.45" accelerate safetensors

In [ ]:
import gc, hashlib, json, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
PROMPTS = ['I went to the store yesterday, but', 'When I opened the refrigerator this morning,', 'The new employee seemed completely normal until', 'Nobody at the restaurant could explain why', 'My neighbor knocked on my door and asked if', 'The package arrived three weeks late, and', 'I knew the hotel was unusual when', 'At first the job interview was going well, but', 'The sign on the door clearly said', 'I called customer service because']

In [ ]:
MODELS = [('small', 'GPT-2 Small', 'openai-community/gpt2'), ('medium', 'GPT-2 Medium', 'openai-community/gpt2-medium'), ('large', 'GPT-2 Large', 'openai-community/gpt2-large'), ('xl', 'GPT-2 XL', 'openai-community/gpt2-xl')]

In [ ]:
SEED = 1337
TEMPERATURE = 0.95
TOP_P = 0.95
TOP_K = 50

STANDARD_WORD_CAP = 300
STANDARD_MAX_NEW_TOKENS = 700

TELEPHONE_WORD_CAP = 300

# IMPORTANT: copy the ORIGINAL bakeoff's scheduling target.
# This remains 800 even though we only DISPLAY the first 300 words.
LEGACY_SCHEDULER_TARGET_WORDS = 800

OUTPUT = Path("joke_bank.json")
CHECKPOINT = Path("joke_bank_checkpoint_v7.json")


## Exact uploaded original references

In [ ]:
LEGACY_ORIGINAL_SMALL = 'I went to the store yesterday, but there was some guy outside that came in and said I could get a new pair and if you give him a dollar, I\'ll work with him."\n\nThe same store had an advertisement for one of D.C.\'s newest boutiques and even more for one of its own.\n\n"You can\'t buy the newest, the cheapest or the newest," Lasseter said. "You\'ve got to go through the new."\n\nLasseter went back on her story, saying she went to the store to get a new pair of black gloves with silver buckles. She asked the clerk if she could try a new pair of new gloves.\n\n"No," Lasseter said, "just you\'re going to have to have at least two. There\'s no doubt about it."\n\nLasseter also said she wasn\'t sure if she\'d ever have any money in a new pair of new gloves.\n\nThe clerk, who also worked for the Washington Metropolitan Area\'s Transportation Department, said she was told that if she could get three of them for $3, she could buy four.\n\n"It\'s a lot cheaper," she said. "It doesn\'t cost anything, doesn\'t break the bank or the day."\n\nLasseter said she\'s very grateful for what her friend, Chris Lasseter, gave her.\n\n"We\'ve gone through this and done a lot," Lasseter said. "It\'s a blessing to get to know one of the most accomplished people in the world. It\'s really been a blessing and I think it\'s a blessing for all of us. We\'re just thankful for how much I\'m helping make this country better."\n\nMore from Post Wire:\n\nThe man who drove an SUV over people\'s heads on Election Day in Virginia was acquitted. But the trial judge, who said he hadn\'t done a good enough job of showing that this was an act of terrorism, also said he has no regrets for what he did.\n\nA group of four people were killed in the attack and the attacker, a 29-year-old D.C. resident named Devin Kelley, confessed. The suspect has been named as 21-year-old D.C. man Aaron Alexis and 29-year-old Washington man Devin Jones, all of whom were married when they were 17.\n\nThe three other suspects in the attack were taken into custody and charged in the rampage that left one woman dead.\n\nAt a news conference, Mayor Muriel Bowser said Wednesday she will be offering up additional safety measures for pedestrians, bicyclists and motorists in D.C.\n\n"There has been a tremendous level of support for the victims of the attack — many of them adults and children," she said.\n\nThe mayor also said Wednesday that D.C. Councilman Jim McDermott, whose district includes the District, and several of the injured will continue to receive treatment.\n\nThe attack on the Washington Metro station will continue, with the last stops being at 12:05 a.m., around 10:06 a.m. and 11:22 p.m. ET.\n\nMetro will be closed for the night and will continue to operate from about 3 a.m. until 10 p.m. ET.\n\nUpdate: 11:26 p.m.: The Metropolitan Police Department said in a statement that it has not received a number of calls about the incident.\n\nOriginal story: The D.C. woman who killed two people on Election Day is cooperating with authorities, a DC police spokesperson said Wednesday morning.\n\n"The public is being urged to remain vigilant in D.C. to ensure that any personal information that should be left with the investigation is safe and confidential," a police spokeswoman said.\n\nThe spokeswoman also said she was unaware of any other calls about the incident.\n\n"We are cooperating fully with the investigation," she said.\n\nUpdate: The Metropolitan Police Department said Wednesday that it has not received a call about the incident.\n\nOriginal story: The Washington Metropolitan Area is working with law enforcement to identify the man who killed two people on Election Day in a parking lot in the city.\n\nA woman was killed while trying to protect her 15-year-old son in the parking lot of D.C.\'s Washington Metro station at about 11:30 p.m. Dec. 6, 2017, police said.\n\nThe man, 31, walked into the parking lot from the front of the station with a pistol to his back and took a driver\'s license, police said.\n\nHe tried to run away, but came to the front of the parking lot and started firing at officers, police said. Then a man drove in front of him, drove down the street, and opened fire and wounded another officer in the thigh.\n\nA second vehicle then hit the man before the officers arrived, police said.\n\nNo one was injured in the incident.\n\nOriginal story: Police said the man who fired the fatal shot killed two people and injured another two in the parking lot of the D.C. Metro station and then ran out of the building and ran to a nearby'

LEGACY_ORIGINAL_LARGE = 'I went to the store yesterday, but the delivery guy took my order while I was waiting for the car to come in and then called to cancel the order. I was worried that they might send me a replacement, but when they came I was pretty disappointed that they decided to put the order on hold for 10 mins before delivering it. I was very upset and disappointed to hear that they couldn\'t even do it as they\'d ordered it. It was a pretty bad experience. They didn\'t even try to make it easy on me.\n\nOrdered 3 items from Walmart online. When I reached Walmart to pick up my order, a young lady asked me if I was at the store yet because I had paid at checkout. When I said I wasn\'t, she told me I would have to wait at the store. So I waited in the store and she asked me to get another item. This was pretty rude and rude, I don\'t think she thought she could get me any other item after that, but I wasn\'t about to complain so I finally left the store.\n\nI\'m still pissed. I ordered this for my sister\'s birthday. The tracking number was sent for $45, which is the lowest I\'ve ever seen it go. I live in Maryland, so it should have come with tracking so that I don\'t have to get it from home. The lady at the checkout said the receipt was received and the package was ready to be picked up, when in fact, she told me it would arrive in about 4-5 days. I can\'t blame her for the time it took because I\'m a bad businessperson and tried everything to get this order out of the way. But I\'m not sure if I\'m supposed to go through all this trouble with a company that doesn\'t bother to communicate. I called Walmart to complain, and they were rude and unresponsive. I called Walmart again and spoke with someone who said "they are trying to do the right thing." The manager there was very rude to me and said he has "no idea how to handle customer service issues." I then decided to just go ahead and buy the item at my local Walmart.\n\nWhat\'s up with Walmart? I received this box at my front door on August 20th. I waited for over 48 hours for the post office to deliver it to me. Finally, on August 26th I got a call from Walmart and the person at the phone said that they had to call the store and see if I really did received it and that they were sorry. I guess I just don\'t give a damn about the person at the front door if they make me wait so long. Walmart, I hope your customer service starts paying attention to the customers not the money line.\n\nI have been a customer since I started working there in 1997 and I have never had a problem or complaint with Walmart before. I was just disappointed that they would do that. Why did I have to wait and worry about this for a second time?\n\nThis one time I am going to tell you that Walmart, even after the delivery was delayed they didn\'t apologize for the wait. After that it\'s up to me to come and pick up the package that was supposed to arrive.\n\nI am a customer of Walmart for 20 years. The quality of my products are amazing! I am never disappointed. And the prices are reasonable. They have an awesome team that care about each and every customer.\n\n\nWalmart is my go to for all my electronics. Thank you.\n\nWalmart is the worst company in the world and their customer service has to go. This happened to me a few weeks ago. The delivery guy was late and told me that he had to wait for his supervisor for the package to be delivered. I was on my way to pick it up and when I came out my supervisor told me it was still a few days before the box was going to get here. At least the delivery person called me before he left and apologized for the wait. I am still upset about that.\n\n\nWalmart is the worst customer service company I have ever encountered.\n\nSo when I picked up my package from walmart today, I was upset. The package was a month late and the only way I can track that is by calling Wal-Mart. The customer service representative was nice and said he needed to pick up the package today. About an hour later I got a call from Walmart and the rep said they needed to wait for a supervisor to'

LEGACY_SMALL_SHA256 = '349e32015a181fa805ed8ddcd1e5c72bfac103d556f6272619437af42cdb220f'
LEGACY_LARGE_SHA256 = 'dbcb1a9db15f36a27614f4ea83c293046c3c89e4a5e67fac3b4087a6369f7c7d'
LEGACY_SMALL_META = {'model_id': 'openai-community/gpt2', 'prompt': 'I went to the store yesterday, but', 'target_words': 800, 'actual_words': 800, 'seed': 1337, 'temperature': 0.95, 'top_p': 0.95, 'top_k': 50, 'native_context_limit': 1024, 'chunks': 7, 'seconds': 20.89195156097412, 'torch_version': '2.11.0+cu128', 'transformers_version': '5.15.0'}
LEGACY_LARGE_META = {'model_id': 'openai-community/gpt2-large', 'prompt': 'I went to the store yesterday, but', 'target_words': 800, 'actual_words': 800, 'seed': 1337, 'temperature': 0.95, 'top_p': 0.95, 'top_k': 50, 'native_context_limit': 1024, 'chunks': 4, 'seconds': 23.475491285324097, 'torch_version': '2.11.0+cu128', 'transformers_version': '5.15.0'}

assert hashlib.sha256(LEGACY_ORIGINAL_SMALL.encode()).hexdigest() == LEGACY_SMALL_SHA256
assert hashlib.sha256(LEGACY_ORIGINAL_LARGE.encode()).hexdigest() == LEGACY_LARGE_SHA256

print("Legacy files embedded and SHA-256 verified.")
print("Original Small metadata:", LEGACY_SMALL_META)
print("Original Large metadata:", LEGACY_LARGE_META)


## Generation helpers

In [ ]:

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def word_count(text):
    return len(re.findall(r"\S+", text))

def cut_to_n_words(text, n):
    matches = list(re.finditer(r"\S+", text))
    if len(matches) <= n:
        return text
    return text[:matches[n - 1].end()]

def load_model(model_id):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
    )
    model.eval()
    return tokenizer, model

def get_context_limit(model, tokenizer):
    candidates = []

    for attr in (
        "max_position_embeddings",
        "n_positions",
        "n_ctx",
        "seq_length",
    ):
        value = getattr(model.config, attr, None)
        if (
            isinstance(value, int)
            and 32 <= value <= 1_000_000
        ):
            candidates.append(value)

    tmax = getattr(tokenizer, "model_max_length", None)
    if (
        isinstance(tmax, int)
        and 32 <= tmax <= 1_000_000
    ):
        candidates.append(tmax)

    return min(candidates) if candidates else 1024

@torch.inference_mode()
def standard_story(tokenizer, model, prompt):
    """
    One normal autocomplete call.
    Natural EOS is enabled.
    We display at most 300 words.
    """
    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    device = next(model.parameters()).device
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    t0 = time.perf_counter()

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=STANDARD_MAX_NEW_TOKENS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    elapsed = time.perf_counter() - t0

    new_ids = outputs[0, input_ids.shape[1]:]
    continuation = tokenizer.decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    text = prompt + continuation

    natural_eos = (
        tokenizer.eos_token_id is not None
        and tokenizer.eos_token_id in new_ids.tolist()
    )

    hit_cap = word_count(text) > STANDARD_WORD_CAP
    if hit_cap:
        text = cut_to_n_words(
            text,
            STANDARD_WORD_CAP,
        )

    return {
        "text": text,
        "generation_type": "standard",
        "actual_words": word_count(text),
        "word_cap": STANDARD_WORD_CAP,
        "natural_eos": bool(natural_eos),
        "stop_reason": (
            "eos"
            if natural_eos
            else "word_cap"
            if hit_cap
            else "token_cap"
        ),
        "generated_tokens": int(len(new_ids)),
        "generation_seconds": elapsed,
    }

@torch.inference_mode()
def original_rolling_telephone(tokenizer, model, prompt):
    """
    THE ORIGINAL BAKEOFF ALGORITHM, adapted only by stopping/displaying
    once 300 words exist.

    Key compatibility properties:
      - same rolling accumulated text
      - same context-window budget logic
      - same max chunk = min(256, context_limit // 4)
      - same original 800-word scheduler target
      - same temperature/top_p/top_k
      - EOS suppressed
      - same decode settings

    We do NOT change the internal target to 300 because that would change
    the second chunk size and could alter the original sample.
    """
    text = prompt
    chunks = 0
    total_generated_tokens = 0
    started = time.perf_counter()

    context_limit = get_context_limit(
        model,
        tokenizer,
    )

    max_chunk = min(
        256,
        max(64, context_limit // 4),
    )

    context_budget = max(
        32,
        context_limit - max_chunk - 8,
    )

    bad_words_ids = None
    if tokenizer.eos_token_id is not None:
        bad_words_ids = [[tokenizer.eos_token_id]]

    chunk_records = []

    while word_count(text) < TELEPHONE_WORD_CAP:
        # ORIGINAL notebook scheduled chunks based on an 800-word target.
        remaining_words = (
            LEGACY_SCHEDULER_TARGET_WORDS
            - word_count(text)
        )

        requested = int(
            remaining_words * 1.45
        ) + 12

        new_tokens = max(
            32,
            min(max_chunk, requested),
        )

        encoded = tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False,
        )["input_ids"]

        # ORIGINAL rolling context behavior.
        input_ids = encoded[:, -context_budget:]

        device = next(model.parameters()).device
        input_ids = input_ids.to(device)

        attention_mask = torch.ones_like(
            input_ids,
            device=device,
        )

        t0 = time.perf_counter()

        # First attempt is EXACTLY the original bakeoff generate() call.
        # This is critical for reproducing the original p01 Small/Large samples.
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=new_tokens,

            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            top_k=TOP_K,
            repetition_penalty=1.0,

            pad_token_id=tokenizer.pad_token_id,
            bad_words_ids=bad_words_ids,
            use_cache=True,
        )

        new_ids = outputs[
            0,
            input_ids.shape[1]:,
        ]

        continuation = tokenizer.decode(
            new_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        # Rare transformers edge case:
        # a chunk can consist only of special/padding tokens and therefore
        # decode to zero visible characters. The original notebook simply
        # crashed here. For the 160-story bank we need to continue.
        #
        # IMPORTANT: this fallback runs ONLY after an empty original attempt.
        # Every normal chunk remains exactly the original algorithm.
        empty_retries = 0
        while not continuation:
            empty_retries += 1

            if empty_retries > 16:
                raise RuntimeError(
                    "Legacy telephone chunk produced no visible text "
                    "after 16 fallback retries."
                )

            print(
                f"    empty legacy chunk; fallback retry "
                f"{empty_retries}/16"
            )

            # Explicitly disable EOS as a stopping condition on the fallback.
            # EOS remains in bad_words_ids as well. This prevents an
            # EOS/pad-only sequence from immediately terminating again.
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=new_tokens,

                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
                repetition_penalty=1.0,

                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=None,
                bad_words_ids=bad_words_ids,
                use_cache=True,
            )

            new_ids = outputs[
                0,
                input_ids.shape[1]:,
            ]

            continuation = tokenizer.decode(
                new_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )

        chunk_seconds = (
            time.perf_counter() - t0
        )

        words_before = word_count(text)
        text += continuation
        words_after = word_count(text)

        chunks += 1
        total_generated_tokens += int(
            len(new_ids)
        )

        chunk_records.append({
            "chunk": chunks,
            "requested_tokens": new_tokens,
            "generated_tokens": int(
                len(new_ids)
            ),
            "words_before": words_before,
            "words_after": words_after,
            "seconds": chunk_seconds,
        })

    # Only difference from original 800-word run:
    # display/save the first 300 words instead.
    text = cut_to_n_words(
        text,
        TELEPHONE_WORD_CAP,
    )

    return {
        "text": text,
        "generation_type": "telephone",
        "telephone_algorithm": "original_rolling_bakeoff",
        "actual_words": word_count(text),
        "word_cap": TELEPHONE_WORD_CAP,
        "eos_suppressed": True,
        "legacy_scheduler_target_words": (
            LEGACY_SCHEDULER_TARGET_WORDS
        ),
        "chunks": chunks,
        "chunk_records": chunk_records,
        "generated_tokens": (
            total_generated_tokens
        ),
        "generation_seconds": (
            time.perf_counter() - started
        ),
        "stop_reason": "300_word_display_cap",
    }

def save_checkpoint(generated):
    CHECKPOINT.write_text(
        json.dumps(
            generated,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )


## Generate all 160 stories

In [ ]:

generated = {}

for difficulty, model_label, model_id in MODELS:
    print("\n" + "=" * 90)
    print(model_label, "-", model_id)
    print("=" * 90)

    tokenizer, model = load_model(model_id)

    # ------------------------------------------------------------
    # STANDARD STREAM
    # ------------------------------------------------------------
    seed_everything(SEED)

    for pi, prompt in enumerate(
        PROMPTS,
        start=1,
    ):
        pid = f"p{pi:02d}"

        for rep in [1, 2]:
            aid = (
                f"{pid}_{difficulty}"
                f"_standard_{rep}"
            )

            result = standard_story(
                tokenizer,
                model,
                prompt,
            )

            generated[aid] = {
                "answer_id": aid,
                "prompt_id": pid,
                "difficulty": difficulty,
                "model_label": model_label,
                "model_id": model_id,
                "replicate": rep,
                "seed": SEED,
                **result,
            }

            save_checkpoint(generated)

            print(
                f"{aid}: "
                f"{result['actual_words']}w, "
                f"{result['stop_reason']}, "
                f"{result['generation_seconds']:.1f}s"
            )

    # ------------------------------------------------------------
    # TELEPHONE STREAM
    # Re-seed so p01 telephone #1 starts exactly like the
    # original bakeoff run for this model.
    # ------------------------------------------------------------
    seed_everything(SEED)

    for pi, prompt in enumerate(
        PROMPTS,
        start=1,
    ):
        pid = f"p{pi:02d}"

        for rep in [1, 2]:
            aid = (
                f"{pid}_{difficulty}"
                f"_telephone_{rep}"
            )

            result = (
                original_rolling_telephone(
                    tokenizer,
                    model,
                    prompt,
                )
            )

            generated[aid] = {
                "answer_id": aid,
                "prompt_id": pid,
                "difficulty": difficulty,
                "model_label": model_label,
                "model_id": model_id,
                "replicate": rep,
                "seed": SEED,
                **result,
            }

            save_checkpoint(generated)

            print(
                f"{aid}: "
                f"{result['actual_words']}w, "
                f"{result['chunks']} chunks, "
                f"{result['generation_seconds']:.1f}s"
            )

    del model, tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTOTAL:", len(generated))
assert len(generated) == 160


## Validate structure + exact original Small/Large prefixes

In [ ]:

# ------------------------------------------------------------
# STRUCTURE
# ------------------------------------------------------------
assert len(generated) == 160

for pi in range(1, 11):
    pid = f"p{pi:02d}"

    prompt_answers = [
        a
        for a in generated.values()
        if a["prompt_id"] == pid
    ]

    std = [
        a
        for a in prompt_answers
        if a["generation_type"]
        == "standard"
    ]

    tel = [
        a
        for a in prompt_answers
        if a["generation_type"]
        == "telephone"
    ]

    assert len(std) == 8
    assert len(tel) == 8

    for group in [std, tel]:
        counts = {}

        for a in group:
            counts[a["difficulty"]] = (
                counts.get(
                    a["difficulty"],
                    0,
                )
                + 1
            )

        assert counts == {
            "small": 2,
            "medium": 2,
            "large": 2,
            "xl": 2,
        }, (pid, counts)

for a in generated.values():
    if a["generation_type"] == "standard":
        assert a["actual_words"] <= 300
    else:
        assert a["actual_words"] == 300
        assert a["eos_suppressed"] is True
        assert (
            a["legacy_scheduler_target_words"]
            == 800
        )

# ------------------------------------------------------------
# EXACT ORIGINAL-SAMPLE PREFIX VALIDATION
# ------------------------------------------------------------
def word_list(text):
    return re.findall(r"\S+", text)

def assert_exact_prefix(
    current,
    original,
    label,
):
    current_words = word_list(current)
    original_words = word_list(original)

    assert (
        current_words
        == original_words[
            :len(current_words)
        ]
    ), (
        f"{label} does not match "
        f"the uploaded original prefix."
    )

    print(
        f"PASS: {label}: "
        f"{len(current_words)} / "
        f"{len(original_words)} words "
        f"match exactly."
    )

assert_exact_prefix(
    generated[
        "p01_small_telephone_1"
    ]["text"],
    LEGACY_ORIGINAL_SMALL,
    "GPT-2 Small original",
)

assert_exact_prefix(
    generated[
        "p01_large_telephone_1"
    ]["text"],
    LEGACY_ORIGINAL_LARGE,
    "GPT-2 Large original",
)

print("\nALL VALIDATION PASSED")
print(
    "Original Small/Large are represented "
    "by their exact first 300 words."
)


## Build the 160-story `joke_bank.json`

In [ ]:

bank = {
    "schema_version": 7,
    "generated": True,

    "generation_settings": {
        "seed": SEED,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,

        "standard_word_cap": (
            STANDARD_WORD_CAP
        ),
        "standard_eos": "natural",

        "telephone_word_cap": (
            TELEPHONE_WORD_CAP
        ),
        "telephone_algorithm": (
            "original_rolling_bakeoff"
        ),
        "telephone_eos": "suppressed",
        "telephone_internal_scheduler_target_words": (
            LEGACY_SCHEDULER_TARGET_WORDS
        ),
    },

    "legacy_validation": {
        "small": {
            "answer_id": (
                "p01_small_telephone_1"
            ),
            "uploaded_sha256": (
                LEGACY_SMALL_SHA256
            ),
            "validation": (
                "exact_first_300_words"
            ),
        },
        "large": {
            "answer_id": (
                "p01_large_telephone_1"
            ),
            "uploaded_sha256": (
                LEGACY_LARGE_SHA256
            ),
            "validation": (
                "exact_first_300_words"
            ),
        },
    },

    "models": [
        {
            "difficulty": d,
            "label": label,
            "model_id": model_id,
        }
        for d, label, model_id
        in MODELS
    ],

    "prompts": [],
}

for pi, prompt in enumerate(
    PROMPTS,
    start=1,
):
    pid = f"p{pi:02d}"

    prompt_answers = sorted(
        [
            a
            for a in generated.values()
            if a["prompt_id"] == pid
        ],
        key=lambda x: x["answer_id"],
    )

    bank["prompts"].append({
        "prompt_id": pid,
        "text": prompt,
        "answers": prompt_answers,
    })

OUTPUT.write_text(
    json.dumps(
        bank,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Wrote",
    OUTPUT,
    "-",
    round(
        OUTPUT.stat().st_size / 1024,
        1,
    ),
    "KB",
)


## Summary

In [ ]:
df = pd.DataFrame(
    list(generated.values())
)

display(
    df.groupby(
        [
            "generation_type",
            "model_label",
        ]
    ).agg(
        stories=(
            "answer_id",
            "count",
        ),
        median_words=(
            "actual_words",
            "median",
        ),
        min_words=(
            "actual_words",
            "min",
        ),
        max_words=(
            "actual_words",
            "max",
        ),
        mean_seconds=(
            "generation_seconds",
            "mean",
        ),
    ).reset_index()
)

print(
    "\nOriginal telephone chunk records:"
)

for aid in [
    "p01_small_telephone_1",
    "p01_large_telephone_1",
]:
    print("\n", aid)
    display(
        pd.DataFrame(
            generated[aid][
                "chunk_records"
            ]
        )
    )


## Download the finished bank

In [ ]:
from google.colab import files
files.download("joke_bank.json")
